<a href="https://colab.research.google.com/github/sousadeoliveiragabriella19-droid/AtividadePratica1/blob/main/AulaPratica01_MetroBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
from collections import deque
import os

# Definições globais para o provedor e modelos de LLM
PROVEDOR = "offline" # Pode ser "groq", "ollama", ou "offline"
MODELO_GROQ = "llama3-8b-8192"
MODELO_OLLAMA = "llama3"

def obter_chave_groq():
  """Busca a chave SEM escrevê-la no código: Colab Secrets → .env → variável de ambiente."""
  try:
    from google.colab import userdata
    return userdata.get("GROQ_API_KEY")
  except Exception:
    pass
  try:
    from dotenv import load_dotenv
    load_dotenv()
  except Exception:
    pass
  return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
  """Envia mensagens ao Llama e devolve o texto da resposta."""
  if PROVEDOR == "groq":
    from groq import Groq
    cliente = Groq(api_key=obter_chave_groq())
    extras = {"response_format": {"type": "json_object"}} if modo_json else {}
    resposta = cliente.chat.completions.create(
      model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
    return resposta.choices[0].message.content
  elif PROVEDOR == "ollama":
    import ollama
    extras = {"format": "json"} if modo_json else {}
    resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
      options={"temperature": 0}, **extras)
    return resposta["message"]["content"]
  else:
    raise RuntimeError("Modo offline: nenhum LLM configurado.")

# ---------- GRAFO ----------
LINHA_1_AZUL = [
"Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
"Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
"São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
"Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
"Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
]

def construir_grafo(estacoes):
  """Cada estação vira um nó ligado à anterior e à próxima da lista."""
  grafo = {estacao: [] for estacao in estacoes}
  for i in range(len(estacoes) - 1):
    a, b = estacoes[i], estacoes[i + 1]
    grafo[a].append(b) # a → b
    grafo[b].append(a) # b → a (o trem anda nos dois sentidos)
  return grafo

GRAFO = construir_grafo(LINHA_1_AZUL)

# ---------- LOCAIS ----------
LOCAIS = {
"Shopping Metrô Tucuruvi": "Tucuruvi",
"Terminal Rodoviário Tietê": "Portuguesa-Tietê",
"Museu de Arte Sacra": "Tiradentes",
"Pinacoteca": "Luz",
"Museu da Língua Portuguesa": "Luz",
"Mosteiro de São Bento": "São Bento",
"Rua 25 de Março": "São Bento",
"Catedral da Sé": "Sé",
"Bairro da Liberdade": "Japão-Liberdade",
"Centro Cultural São Paulo": "Vergueiro",
"Shopping Metrô Santa Cruz": "Santa Cruz",
"Universidade São Judas": "São Judas",
"Terminal Rodoviário Jabaquara": "Jabaquara",
}
# ---------- BFS ----------
def reconstruir_caminho(pai, destino):
  """Puxa o 'fio de Ariadne': do destino até a origem, depois inverte."""
  caminho = []
  atual = destino
  while atual is not None:
    caminho.append(atual)
    atual = pai[atual]
  return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):

  if origem in bloqueadas or destino in bloqueadas:
    return None, []
  fila = deque([origem])
  pai = {origem: None}
  ordem_visita = []
  while fila:
    atual = fila.popleft() # 1º da fila sai primeiro (FIFO)
    ordem_visita.append(atual)
    if atual == destino:
      return reconstruir_caminho(pai, destino), ordem_visita
    for vizinho in grafo[atual]:
      if vizinho not in pai and vizinho not in bloqueadas:
        pai[vizinho] = atual
        fila.append(vizinho)
  return None, ordem_visita
# ---------- DFS ----------

def dfs(grafo, origem, destino, bloqueadas=()):
  """Vai fundo no 1º vizinho; se não achar, volta (backtracking) e tenta o próximo."""
  if origem in bloqueadas or destino in bloqueadas:
    return None, []
  visitados = set()
  ordem_visita = []

  def explorar(atual, caminho):
    visitados.add(atual)
    ordem_visita.append(atual)
    if atual == destino:
      return caminho
    for vizinho in grafo[atual]:
      if vizinho not in visitados and vizinho not in bloqueadas:
        resultado = explorar(vizinho, caminho + [vizinho]) # mergulha
        if resultado:
          return resultado
    return None # beco sem saída → volta

  return explorar(origem, [origem]), ordem_visita
# ---------- LÓGICA PROPOSICIONAL ----------
def pode_embarcar(P, Q, R):
  """P: estação aberta | Q: precisa de acessibilidade | R: elevador funcionando"""
  return P and ((not Q) or R)
def tabela_verdade():
  print(" P | Q | R | P ∧ (¬Q ∨ R)")
  print("-" * 40)
  for P, Q, R in product([True, False], repeat=3):
    print(f" {P!s:5} | {Q!s:5} | {R!s:5} | {pode_embarcar(P, Q, R)}")
# ---------- LÓGICA DE PRIMEIRA ORDEM ----------
def fatos_base():
  """Fatos fixos do mundo: quais estações existem e o que fica perto de cada uma."""
  fatos = set()
  for estacao in LINHA_1_AZUL:
    fatos.add(("estacao", estacao))
  for local, estacao in LOCAIS.items():
    fatos.add(("proximo_de", local, estacao))
  return fatos
def consultar(fatos, predicado):
  """Devolve os argumentos de todos os fatos de um predicado. Ex.: consultar(f, 'destino') → [('Luz',)]"""
  return [f[1:] for f in fatos if f[0] == predicado]
def r_origem(fatos):
  novos = set()
  for (local,) in consultar(fatos, "usuario_esta_em"):
    for (l, e) in consultar(fatos, "proximo_de"):
      if l == local:
        novos.add(("origem", e))
  for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
    novos.add(("origem", e))
  return novos
def r_destino(fatos):
  novos = set()
  for (local,) in consultar(fatos, "usuario_quer_ir"):
    for (l, e) in consultar(fatos, "proximo_de"):
      if l == local:
        novos.add(("destino", e))
  for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
    novos.add(("destino", e))
  return novos
def r_bloqueio(fatos):
  return {( "bloqueada", e) for (e,) in consultar(fatos, "fechada")}
def r_acessibilidade(fatos):
  if not consultar(fatos, "precisa_acessibilidade"):
    return set()
  return {( "inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}
def r_alerta(fatos):
  novos = set()
  inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
  for papel in ("origem", "destino"):
    for (e,) in consultar(fatos, papel):
      if e in inacessiveis:
        novos.add(("alerta", papel, e))
  return novos

REGRAS = [
("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
("R4 acessibilidade","∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),

("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta),
]
def encadear_para_frente(fatos, regras, verbose=False):
  """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
  fatos = set(fatos)
  justificativas = {}
  rodada = 0
  while True:
    rodada += 1
    novos_na_rodada = set()
    for nome, _formula, regra in regras:
      for fato in regra(fatos) - fatos:
        novos_na_rodada.add(fato)
        justificativas[fato] = nome
    if verbose:
      print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
    if not novos_na_rodada:
      return fatos, justificativas
    fatos |= novos_na_rodada
# ---------- PLANEJADOR ----------
TEMPO_POR_TRECHO = 2 # minutos por trecho (valor simulado, didático)
def planejar(pedido, fechadas=(), manutencao=(), algoritmo="BFS"):
  """pedido = {"origem": (tipo, nome), "destino": (tipo, nome), "acessibilidade": bool}
  tipo é "local" ou "estacao"."""
# 1) Monta a base de conhecimento com o pedido e o cenário
  fatos = fatos_base()
  tipo_o, nome_o = pedido["origem"]
  tipo_d, nome_d = pedido["destino"]
  fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
  fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))
  if pedido.get("acessibilidade"):
    fatos.add(("precisa_acessibilidade",))
  for e in fechadas:
    fatos.add(("fechada", e))
  for e in manutencao:
    fatos.add(("elevador_em_manutencao", e))
# 2) Inferência lógica
  fatos, justificativas = encadear_para_frente(fatos, REGRAS)
  origem = consultar(fatos, "origem")[0][0]
  destino = consultar(fatos, "destino")[0][0]
  bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
  alertas = consultar(fatos, "alerta")
# 3) Busca usando o que a lógica deduziu
  buscar = bfs if algoritmo == "BFS" else dfs
  caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)
  return {
  "origem": origem, "destino": destino, "algoritmo": algoritmo,
  "caminho": caminho, "visitados": visitados,
  "bloqueadas": sorted(bloqueadas),
  "alertas": [f"{papel}: {e}" for papel, e in alertas],
  "paradas": len(caminho) - 1 if caminho else None,
  "tempo_min": (len(caminho) - 1) * TEMPO_POR_TRECHO if caminho else None,
  "regras_usadas": sorted(set(justificativas.values())),
}
# ---------- INTÉRPRETE ----------
import unicodedata
import re
import json
from itertools import product

def normalizar(texto):
  """Minúsculas e sem acentos: 'São Bento' → 'sao bento'."""
  texto = unicodedata.normalize("NFD", texto.lower())
  return "".join(c for c in texto if unicodedata.category(c) != "Mn")
def resolver_nome(nome):
  """GUARDRAIL: só aceita nomes que existem de verdade. Senão, None."""
  if not nome:
    return None
  alvo = normalizar(nome).strip()
  for estacao in LINHA_1_AZUL:
    if normalizar(estacao) == alvo:
      return ("estacao", estacao)
  for local in LOCAIS:
    if normalizar(local) == alvo:
      return ("local", local)
  return None
PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP.
Sua única tarefa é transformar o pedido do passageiro em JSON.
Estações válidas: {estacoes}
Locais válidos: {locais}
Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
"destino": "<nome exato de estação ou local, ou null>",
"acessibilidade": <true ou false>}}
Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas,
mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se não souber algum campo, use null. Nunca invente nomes."""
def interpretar_offline(texto):
  """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""
  texto_min = texto.lower()
  texto_sem = normalizar(texto) # mesmo tamanho, só sem acentos
  candidatos = [(n, "estacao") for n in LINHA_1_AZUL] + [(n, "local") for n in LOCAIS]
  candidatos.sort(key=lambda c: len(c[0]), reverse=True) # nomes longos primeiro
  ocupado = [False] * len(texto_min)
  encontrados = []
  for nome, tipo in candidatos:
    buscas = [(texto_min, nome.lower())]
    if len(nome) > 4 and len(texto_sem) == len(texto_min):
      buscas.append((texto_sem, normalizar(nome))) # aceita "se" sem acento só p/ nomes longos
    for base, padrao in buscas:
      for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
        if not any(ocupado[m.start():m.end()]):
          encontrados.append((m.start(), nome))
          for i in range(m.start(), m.end()):
            ocupado[i] = True
  encontrados.sort()
  palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade",
  "muleta", "carrinho de bebe", "elevador"]
  return {
  "origem": encontrados[0][1] if len(encontrados) > 0 else None,
  "destino": encontrados[1][1] if len(encontrados) > 1 else None,
  "acessibilidade": any(p in texto_sem for p in palavras_acess),
  }
def interpretar_pedido(texto):
  """Texto livre → pedido validado. Usa o Llama; se falhar, cai no modo offline."""
  if PROVEDOR == "offline":
    bruto = interpretar_offline(texto)
    fonte = "offline"
  else:
    sistema = PROMPT_INTERPRETE.format(
    estacoes=", ".join(LINHA_1_AZUL), locais=", ".join(LOCAIS))
    try:
      resposta = chamar_llm([{"role": "system", "content": sistema},
      {"role": "user", "content": texto}], modo_json=True)
      bruto = json.loads(resposta)
      fonte = PROVEDOR
    except Exception as erro:
      print(f" LLM indisponível ({erro}). Usando modo offline.")
      bruto = interpretar_offline(texto)
      fonte = "offline"
  origem = resolver_nome(bruto.get("origem"))
  destino = resolver_nome(bruto.get("destino"))
  if origem is None or destino is None:
    return None, f"Não entendi origem/destino (resposta bruta: {bruto})"
  pedido = {"origem": origem, "destino": destino,
  "acessibilidade": bool(bruto.get("acessibilidade"))}
  return pedido, f"Interpretado via {fonte}"
# ---------- NARRADOR ----------
def narrar_offline(r):
  if r["caminho"] is None:
    return (f"Não existe rota de {r['origem']} até {r['destino']} "
f"com as estações bloqueadas: {', '.join(r['bloqueadas'])}.")
  texto = (f"Embarque em {r['origem']} e siga pela Linha 1-Azul até {r['destino']}: "
f"{r['paradas']} parada(s), cerca de {r['tempo_min']} minutos.")
  if r["alertas"]:
    texto += " Atenção: " + "; ".join(r["alertas"]) + " (elevador em manutenção)."
  return texto
PROMPT_NARRADOR = """Você é o NARRADOR do MetrôBot SP. Explique a rota ao passageiro
em português, em no máximo 4 frases curtas e simpáticas.
Use SOMENTE os dados do JSON. Não invente horários, linhas, estações
ou atrações. Se "caminho" for null, explique que não há rota e cite as
estacões bloqueadas. Se houver "alertas", destaque-os."""
def narrar(resultado):
  """Transforma o resultado da busca em explicação amigável."""
  dados = {k: resultado[k] for k in
("origem", "destino", "caminho", "paradas", "tempo_min", "bloqueadas", "alertas")}
  if PROVEDOR == "offline":
    return narrar_offline(resultado)
  try:
    return chamar_llm([{"role": "system", "content": PROMPT_NARRADOR},
  {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}])
  except Exception as erro:
    return narrar_offline(resultado) + f" (narrador offline: {erro})"
# ---------- VISUALIZAÇÃO ----------
def desenhar_linha(resultado):
  caminho = set(resultado["caminho"] or [])
  visitados = set(resultado["visitados"])
  bloqueadas = set(resultado["bloqueadas"])
  linhas_html = []
  for estacao in LINHA_1_AZUL:
    if estacao in bloqueadas:
      cor, marca = "#d32f2f", " bloqueada"
    elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
      cor, marca = "#0d47a1", " " + ("origem" if estacao == resultado["origem"] else "destino")
    elif estacao in caminho:
      cor, marca = "#1e88e5", "rota"
    elif estacao in visitados:
      cor, marca = "#9e9e9e", "visitada pela busca"
    else:
      cor, marca = "#e0e0e0", ""
    linhas_html.append(
    f"<div style='display:flex;align-items:center;gap:8px;font-family:sans-serif;font-size:13px'>"
f"<span style='display:inline-block;width:14px;height:14px;border-radius:50%;background:{cor}'></span>"
f"<span style='min-width:150px'>{estacao}</span><span style='color:#666'>{marca}</span></div>")
  return "<div style='border-left:4px solid #1e88e5;padding-left:8px'>" + "".join(linhas_html) + "</div>"
# ---------- INTERFACE ----------
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
opcoes = ([(f" {local}", ("local", local)) for local in LOCAIS] +
[(f" {estacao}", ("estacao", estacao)) for estacao in LINHA_1_AZUL])
txt_pedido = widgets.Textarea(placeholder="Ex.: Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara",layout=widgets.Layout(width="95%", height="60px"))
bn_interpretar = widgets.Button(description=" Interpretar pedido", button_style="info")
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:")
dd_destino = widgets.Dropdown(options=opcoes, value=("local", "Terminal Rodoviário Jabaquara"), description="Destino:")
chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")
sel_fechadas = widgets.SelectMultiple(options=LINHA_1_AZUL, description="Fechadas:", rows=5)
sel_manut = widgets.SelectMultiple(options=LINHA_1_AZUL, description="Elevador :", rows=5)
rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
bn_buscar = widgets.Button(description=" Buscar rota", button_style="success")
saida = widgets.Output()
def ao_interpretar(_):
  with saida:
    clear_output()
  pedido, msg = interpretar_pedido(txt_pedido.value)
  print(msg)
  if pedido:
    dd_origem.value = pedido["origem"]
    dd_destino.value = pedido["destino"]
    chk_acess.value = pedido["acessibilidade"]
    print(" Campos preenchidos. Confira e clique em 'Buscar rota'.")
def ao_buscar(_):
  with saida:
    clear_output()
  pedido = {"origem": dd_origem.value, "destino": dd_destino.value,
"acessibilidade": chk_acess.value}
  r = planejar(pedido, sel_fechadas.value, sel_manut.value, rb_algoritmo.value)
  display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} → {r['destino']}</h4>"))
  print(" ", narrar(r))
  print(f" Estações visitadas pela busca: {len(r['visitados'])}")
  print(f" Regras disparadas: {', '.join(r['regras_usadas'])}")
  display(HTML(desenhar_linha(r)))
bn_interpretar.on_click(ao_interpretar)
bn_buscar.on_click(ao_buscar)
painel = widgets.VBox([
widgets.HTML("<h3> MetrôBot SP — Linha 1-Azul</h3>"),
txt_pedido, bn_interpretar,
widgets.HBox([dd_origem, dd_destino]),
widgets.HBox([chk_acess, rb_algoritmo]),
widgets.HBox([sel_fechadas, sel_manut]),
bn_buscar, saida,
])
# ---------- TESTES ----------
def rodar_testes():
# 1. A Linha 1 tem 23 estações e as pontas têm só 1 vizinho
  assert len(GRAFO) == 23
  assert len(GRAFO["Tucuruvi"]) == 1 and len(GRAFO["Jabaquara"]) == 1

# 2. BFS e DFS encontram o mesmo caminho numa linha reta
  c_bfs, _ = bfs(GRAFO, "Sé", "Vergueiro")
  c_dfs, _ = dfs(GRAFO, "Sé", "Vergueiro")
  assert c_bfs == c_dfs == ["Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro"]
# 3. Estação fechada no meio do caminho = sem rota
  caminho, _ = bfs(GRAFO, "Sé", "Jabaquara", bloqueadas={"Paraíso"})
  assert caminho is None
# 4. Regra R2: local conhecido vira estação de destino
  r = planejar({"origem": ("local", "Catedral da Sé"),
"destino": ("local", "Pinacoteca")})
  assert r["destino"] == "Luz" and r["paradas"] == 2
# 5. Regras R4 + R5: acessibilidade + elevador em manutenção = alerta
  r = planejar({"origem": ("estacao", "Sé"), "destino": ("estacao", "Luz"),
"acessibilidade": True}, manutencao=["Luz"])
  assert "destino: Luz" in r["alertas"]
# 6. Passar POR uma estação sem elevador não gera alerta
  r = planejar({"origem": ("estacao", "Sé"), "destino": ("estacao", "Tiradentes"),
"acessibilidade": True}, manutencao=["Luz"])
  assert r["alertas"] == [] and "Luz" in r["caminho"]
  print(" Todos os 6 testes passaram!")
rodar_testes()
# ---------- MOSTRAR O APP ----------
display(painel)

 Todos os 6 testes passaram!


DESAFIO — MetrôBot SP: Linhas 1, 2 e 3

In [7]:
from collections import deque

from itertools import product

import html

import json

import re

import unicodedata

LINHAS = {

    "Linha 1-Azul": [

        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",

        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",

        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",

        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",

        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",

    ],

    "Linha 2-Verde": [

        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",

        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",

        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",

        "Vila Prudente",

    ],

    "Linha 3-Vermelha": [

        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",

        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",

        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",

        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",

        "Artur Alvim", "Corinthians-Itaquera",

    ],

}

CORES = {

    "Linha 1-Azul": "#1e88e5",

    "Linha 2-Verde": "#2e7d32",

    "Linha 3-Vermelha": "#d32f2f",

}

assert sum(len(v) for v in LINHAS.values()) == 55

LOCAIS = {

    "Pinacoteca": "Luz",

    "Catedral da Sé": "Sé",

    "Terminal Rodoviário Jabaquara": "Jabaquara",

    "MASP": "Trianon-Masp",

    "Hospital das Clínicas": "Clínicas",

    "Aquário de São Paulo": "Santos-Imigrantes",

    "Memorial da América Latina": "Palmeiras-Barra Funda",

    "Museu da Imigração": "Brás",

    "Neo Química Arena": "Corinthians-Itaquera",

}

def construir_grafo_multilinhas(linhas):

    grafo = {}

    linhas_do_trecho = {}

    for nome_linha, estacoes in linhas.items():

        for estacao in estacoes:

            grafo.setdefault(estacao, [])

        for a, b in zip(estacoes, estacoes[1:]):

            if b not in grafo[a]:

                grafo[a].append(b)

            if a not in grafo[b]:

                grafo[b].append(a)

            linhas_do_trecho.setdefault((a, b), set()).add(nome_linha)

            linhas_do_trecho.setdefault((b, a), set()).add(nome_linha)

    return grafo, linhas_do_trecho

GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)

assert len(GRAFO) == 52

assert len(GRAFO["Paraíso"]) >= 2

assert "Linha 1-Azul" in LINHAS_DO_TRECHO[("Paraíso", "Ana Rosa")]

assert "Linha 2-Verde" in LINHAS_DO_TRECHO[("Paraíso", "Ana Rosa")]

def reconstruir_caminho(pai, destino):

    caminho = []

    atual = destino

    while atual is not None:

        caminho.append(atual)

        atual = pai[atual]

    return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):

    bloqueadas = set(bloqueadas)

    if origem in bloqueadas or destino in bloqueadas:

        return None, []

    fila = deque([origem])

    pai = {origem: None}

    ordem_visita = []

    while fila:

        atual = fila.popleft()

        ordem_visita.append(atual)

        if atual == destino:

            return reconstruir_caminho(pai, destino), ordem_visita

        for vizinho in grafo[atual]:

            if vizinho not in pai and vizinho not in bloqueadas:

                pai[vizinho] = atual

                fila.append(vizinho)

    return None, ordem_visita

def dfs(grafo, origem, destino, bloqueadas=()):

    bloqueadas = set(bloqueadas)

    if origem in bloqueadas or destino in bloqueadas:

        return None, []

    visitados = set()

    ordem_visita = []

    def explorar(atual, caminho):

        visitados.add(atual)

        ordem_visita.append(atual)

        if atual == destino:

            return caminho

        for vizinho in grafo[atual]:

            if vizinho not in visitados and vizinho not in bloqueadas:

                resultado = explorar(vizinho, caminho + [vizinho])

                if resultado is not None:

                    return resultado

        return None

    return explorar(origem, [origem]), ordem_visita

def contar_baldeacoes(caminho, linhas_do_trecho):

    if not caminho or len(caminho) < 2:

        return 0, []

    primeiro_trecho = linhas_do_trecho[(caminho[0], caminho[1])]

    estados = {

        linha: (0, [linha])

        for linha in primeiro_trecho

    }

    for i in range(1, len(caminho) - 1):

        estacao_troca = caminho[i]

        opcoes = linhas_do_trecho[(caminho[i], caminho[i + 1])]

        novos = {}

        for linha_atual, (trocas, sequencia) in estados.items():

            for nova_linha in opcoes:

                novas_trocas = trocas + (0 if nova_linha == linha_atual else 1)

                nova_seq = sequencia + [nova_linha]

                candidato = (novas_trocas, nova_seq)

                if nova_linha not in novos or candidato[0] < novos[nova_linha][0]:

                    novos[nova_linha] = candidato

        estados = novos

    melhor_linha, (melhor_trocas, sequencia) = min(

        estados.items(),

        key=lambda item: (item[1][0], item[1][1])

    )

    baldeacoes = []

    for i in range(1, len(sequencia)):

        if sequencia[i] != sequencia[i - 1]:

            baldeacoes.append((caminho[i], sequencia[i]))

    return melhor_trocas, baldeacoes

def fatos_base():

    fatos = set()

    for linha, estacoes in LINHAS.items():

        for estacao in estacoes:

            fatos.add(("estacao", estacao))

            fatos.add(("pertence", estacao, linha))

    for local, estacao in LOCAIS.items():

        fatos.add(("proximo_de", local, estacao))

    return fatos

def consultar(fatos, predicado):

    return [f[1:] for f in fatos if f[0] == predicado]

def r_origem(fatos):

    novos = set()

    for (local,) in consultar(fatos, "usuario_esta_em"):

        for l, e in consultar(fatos, "proximo_de"):

            if l == local:

                novos.add(("origem", e))

    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):

        novos.add(("origem", e))

    return novos

def r_destino(fatos):

    novos = set()

    for (local,) in consultar(fatos, "usuario_quer_ir"):

        for l, e in consultar(fatos, "proximo_de"):

            if l == local:

                novos.add(("destino", e))

    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):

        novos.add(("destino", e))

    return novos

def r_bloqueio(fatos):

    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):

    if not consultar(fatos, "precisa_acessibilidade"):

        return set()

    return {

        ("inacessivel", e)

        for (e,) in consultar(fatos, "elevador_em_manutencao")

    }

def r_alerta(fatos):

    novos = set()

    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}

    for papel in ("origem", "destino"):

        for (e,) in consultar(fatos, papel):

            if e in inacessiveis:

                novos.add(("alerta", papel, e))

    return novos

def r_integracao(fatos):

    por_estacao = {}

    for estacao, linha in consultar(fatos, "pertence"):

        por_estacao.setdefault(estacao, set()).add(linha)

    return {

        ("integracao", estacao)

        for estacao, linhas in por_estacao.items()

        if len(linhas) >= 2

    }

def r_linha_paralisada(fatos):

    linhas_paralisadas = {l for (l,) in consultar(fatos, "linha_paralisada")}

    pertencimentos = consultar(fatos, "pertence")

    return {

        ("bloqueada", estacao)

        for estacao, linha in pertencimentos

        if linha in linhas_paralisadas

    }

REGRAS = [

    ("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),

    ("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),

    ("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),

    ("R4 acessibilidade",

     "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))",

     r_acessibilidade),

    ("R5 alerta",

     "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))",

     r_alerta),

    ("R6 integração",

     "∀e ∀l1 ∀l2 ((pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2) → integracao(e))",

     r_integracao),

    ("R7 linha paralisada",

     "∀l (linha_paralisada(l) → ∀e (pertence(e,l) → bloqueada(e)))",

     r_linha_paralisada),

]

def encadear_para_frente(fatos, regras=REGRAS, verbose=False):

    fatos = set(fatos)

    justificativas = {}

    rodada = 0

    while True:

        rodada += 1

        novos_na_rodada = set()

        for nome, _formula, regra in regras:

            for fato in regra(fatos) - fatos:

                novos_na_rodada.add(fato)

                justificativas[fato] = nome

        if verbose:

            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")

        if not novos_na_rodada:

            return fatos, justificativas

        fatos |= novos_na_rodada

TEMPO_POR_TRECHO = 2

TEMPO_POR_BALDEACAO = 5

def planejar(

    pedido,

    fechadas=(),

    manutencao=(),

    algoritmo="BFS",

    linhas_paralisadas=(),

):

    fatos = fatos_base()

    tipo_o, nome_o = pedido["origem"]

    tipo_d, nome_d = pedido["destino"]

    if tipo_o == "local":

        fatos.add(("usuario_esta_em", nome_o))

    else:

        fatos.add(("usuario_esta_na_estacao", nome_o))

    if tipo_d == "local":

        fatos.add(("usuario_quer_ir", nome_d))

    else:

        fatos.add(("usuario_quer_ir_estacao", nome_d))

    if pedido.get("acessibilidade"):

        fatos.add(("precisa_acessibilidade",))

    for e in fechadas:

        fatos.add(("fechada", e))

    for e in manutencao:

        fatos.add(("elevador_em_manutencao", e))

    for linha in linhas_paralisadas:

        fatos.add(("linha_paralisada", linha))

    fatos, justificativas = encadear_para_frente(fatos)

    origens = consultar(fatos, "origem")

    destinos = consultar(fatos, "destino")

    if not origens or not destinos:

        return None

    origem = origens[0][0]

    destino = destinos[0][0]

    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}

    alertas = consultar(fatos, "alerta")

    buscar = bfs if algoritmo.upper() == "BFS" else dfs

    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)

    if caminho:

        n_baldeacoes, pontos_baldeacao = contar_baldeacoes(

            caminho, LINHAS_DO_TRECHO

        )

        paradas = len(caminho) - 1

        tempo = paradas * TEMPO_POR_TRECHO + n_baldeacoes * TEMPO_POR_BALDEACAO

    else:

        n_baldeacoes = 0

        pontos_baldeacao = []

        paradas = None

        tempo = None

    return {

        "origem": origem,

        "destino": destino,

        "algoritmo": algoritmo.upper(),

        "caminho": caminho,

        "visitados": visitados,

        "bloqueadas": sorted(bloqueadas),

        "alertas": sorted(f"{papel}: {e}" for papel, e in alertas),

        "paradas": paradas,

        "baldeacoes": n_baldeacoes,

        "onde_baldear": pontos_baldeacao,

        "tempo_min": tempo,

        "regras_usadas": sorted(set(justificativas.values())),

        "integracoes_deduzidas": sorted(

            e for (e,) in consultar(fatos, "integracao")

        ),

        "fatos": fatos,

        "justificativas": justificativas,

    }

PROVEDOR = "offline"

MODELO_GROQ = "llama-3.3-70b-versatile"

MODELO_OLLAMA = "llama3.2"

def normalizar(texto):

    texto = unicodedata.normalize("NFD", texto.lower())

    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):

    if not nome:

        return None

    alvo = normalizar(nome).strip()

    for estacao in GRAFO:

        if normalizar(estacao) == alvo:

            return ("estacao", estacao)

    for local in LOCAIS:

        if normalizar(local) == alvo:

            return ("local", local)

    return None

def obter_chave_groq():

    try:

        from google.colab import userdata

        return userdata.get("GROQ_API_KEY")

    except Exception:

        pass

    try:

        from dotenv import load_dotenv

        load_dotenv()

    except Exception:

        pass

    import os

    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):

    if PROVEDOR == "groq":

        from groq import Groq

        cliente = Groq(api_key=obter_chave_groq())

        extras = {"response_format": {"type": "json_object"}} if modo_json else {}

        resposta = cliente.chat.completions.create(

            model=MODELO_GROQ,

            messages=mensagens,

            temperature=0,

            **extras,

        )

        return resposta.choices[0].message.content

    if PROVEDOR == "ollama":

        import ollama

        extras = {"format": "json"} if modo_json else {}

        resposta = ollama.chat(

            model=MODELO_OLLAMA,

            messages=mensagens,

            options={"temperature": 0},

            **extras,

        )

        return resposta["message"]["content"]

    raise RuntimeError("Modo offline: nenhum LLM configurado.")

PROMPT_INTERPRETE = """

Você é o módulo de INTERPRETAÇÃO do MetrôBot SP 2.0.

Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas:

{estacoes}

Locais válidos:

{locais}

Responda APENAS com:

{{

  "origem": "<nome exato ou null>",

  "destino": "<nome exato ou null>",

  "acessibilidade": true ou false

}}

Regras:

- Use SOMENTE nomes presentes nas listas.

- Não invente estação ou local.

- Se não souber origem ou destino, use null.

- Acessibilidade = true se mencionar cadeira de rodas, mobilidade reduzida,

  muletas, carrinho de bebê, elevador ou necessidade de acessibilidade.

"""

def interpretar_offline(texto):

    texto_min = texto.lower()

    texto_sem = normalizar(texto)

    candidatos = (

        [(n, "estacao") for n in GRAFO]

        + [(n, "local") for n in LOCAIS]

    )

    candidatos.sort(key=lambda x: len(x[0]), reverse=True)

    ocupados = [False] * len(texto_min)

    encontrados = []

    for nome, _tipo in candidatos:

        for base, padrao in (

            (texto_min, nome.lower()),

            (texto_sem, normalizar(nome)),

        ):

            if not padrao:

                continue

            for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):

                if not any(ocupados[m.start():m.end()]):

                    encontrados.append((m.start(), nome))

                    for i in range(m.start(), m.end()):

                        if i < len(ocupados):

                            ocupados[i] = True

    encontrados.sort()

    palavras_acess = [

        "cadeira de rodas", "acessibilidade", "mobilidade",

        "muleta", "carrinho de bebe", "elevador",

    ]

    return {

        "origem": encontrados[0][1] if len(encontrados) > 0 else None,

        "destino": encontrados[1][1] if len(encontrados) > 1 else None,

        "acessibilidade": any(p in texto_sem for p in palavras_acess),

    }

def interpretar_pedido(texto):

    if PROVEDOR == "offline":

        bruto = interpretar_offline(texto)

        fonte = "offline"

    else:

        sistema = PROMPT_INTERPRETE.format(

            estacoes=", ".join(GRAFO.keys()),

            locais=", ".join(LOCAIS.keys()),

        )

        try:

            resposta = chamar_llm(

                [

                    {"role": "system", "content": sistema},

                    {"role": "user", "content": texto},

                ],

                modo_json=True,

            )

            bruto = json.loads(resposta)

            fonte = PROVEDOR

        except Exception as erro:

            print(f"LLM indisponível ({erro}). Usando modo offline.")

            bruto = interpretar_offline(texto)

            fonte = "offline"

    origem = resolver_nome(bruto.get("origem"))

    destino = resolver_nome(bruto.get("destino"))

    if origem is None or destino is None:

        return None, f"Não entendi origem/destino. Resposta: {bruto}"

    pedido = {

        "origem": origem,

        "destino": destino,

        "acessibilidade": bool(bruto.get("acessibilidade")),

    }

    return pedido, f"Interpretado via {fonte}"

def narrar(resultado):

    if resultado is None:

        return "Não foi encontrada uma rota com as condições informadas."

    if resultado["caminho"] is None:

        bloqueios = ", ".join(resultado["bloqueadas"]) or "nenhum"

        return (

            f"Não há rota de {resultado['origem']} para {resultado['destino']} "

            f"com os bloqueios atuais ({bloqueios})."

        )

    caminho = resultado["caminho"]

    partes = [

        f"Embarque em {resultado['origem']}.",

        "Rota: " + " → ".join(caminho) + ".",

    ]

    if resultado["onde_baldear"]:

        texto_b = "; ".join(

            f"na {estacao}, troque para a {linha}"

            for estacao, linha in resultado["onde_baldear"]

        )

        partes.append("Baldeações: " + texto_b + ".")

    else:

        partes.append("Não há baldeações.")

    partes.append(

        f"São {resultado['paradas']} paradas e tempo estimado de "

        f"{resultado['tempo_min']} minutos."

    )

    if resultado["alertas"]:

        partes.append("Alertas: " + "; ".join(resultado["alertas"]) + ".")

    return " ".join(partes)

def narrar_com_llm(resultado):

    if PROVEDOR == "offline":

        return narrar(resultado)

    prompt = """

Você é o narrador do MetrôBot.

Explique a rota usando SOMENTE os dados do JSON.

Não invente estações, tempos, linhas ou baldeações.

Se houver baldeação, cite exatamente a estação e a nova linha.

"""

    try:

        resposta = chamar_llm([

            {"role": "system", "content": prompt},

            {"role": "user", "content": json.dumps(resultado, ensure_ascii=False)},

        ])

        return resposta

    except Exception:

        return narrar(resultado)

def pode_embarcar(P, Q, R):

    return P and ((not Q) or R)

def tabela_verdade():

    linhas = []

    for P, Q, R in product([True, False], repeat=3):

        linhas.append((P, Q, R, pode_embarcar(P, Q, R)))

    print(" P     Q     R     P ∧ (¬Q ∨ R)")

    print("-" * 36)

    for P, Q, R, resultado in linhas:

        print(f"{str(P):5} {str(Q):5} {str(R):5} {resultado}")

    return linhas

def _cor_estacao(estacao, rota, visitados, bloqueadas):

    if estacao in bloqueadas:

        return "#616161"

    if estacao in rota:

        return "#ff9800"

    if estacao in visitados:

        return "#9e9e9e"

    return "#ffffff"

def _sequencia_linhas_caminho(caminho):

    if not caminho or len(caminho) < 2:

        return []

    primeiro = sorted(LINHAS_DO_TRECHO[(caminho[0], caminho[1])])

    estados = {linha: (0, [linha]) for linha in primeiro}

    for i in range(1, len(caminho) - 1):

        opcoes = sorted(LINHAS_DO_TRECHO[(caminho[i], caminho[i + 1])])

        novos = {}

        for linha_atual, (trocas, sequencia) in estados.items():

            for nova_linha in opcoes:

                candidato = (trocas + (nova_linha != linha_atual), sequencia + [nova_linha])

                if nova_linha not in novos or candidato < novos[nova_linha]:

                    novos[nova_linha] = candidato

        estados = novos

    return min(estados.values(), key=lambda item: (item[0], item[1]))[1]

def desenhar_mini_mapa_rota(resultado):

    caminho = resultado.get("caminho") or []

    if len(caminho) < 2:

        return ""

    sequencia = _sequencia_linhas_caminho(caminho)

    grupos = []

    inicio = 0

    for i in range(1, len(sequencia) + 1):

        if i == len(sequencia) or sequencia[i] != sequencia[inicio]:

            linha = sequencia[inicio]

            estacoes = caminho[inicio:i + 1]

            grupos.append((linha, estacoes))

            inicio = i

    blocos = []

    for indice, (linha, estacoes) in enumerate(grupos):

        cor = CORES[linha]

        estacoes_html = []

        for pos, estacao in enumerate(estacoes):

            primeira = indice == 0 and pos == 0

            ultima = indice == len(grupos) - 1 and pos == len(estacoes) - 1

            integracao = estacao in {e for e, _ in resultado.get("onde_baldear", [])}

            if primeira:

                marcador = "<span style='font-size:16px'>●</span>"

                rotulo = " <span style='font-size:11px;color:#16a34a;font-weight:700'>ORIGEM</span>"

            elif ultima:

                marcador = "<span style='font-size:16px'>●</span>"

                rotulo = " <span style='font-size:11px;color:#dc2626;font-weight:700'>DESTINO</span>"

            elif integracao:

                marcador = "<span style='font-size:17px'>◎</span>"

                rotulo = " <span style='font-size:11px;color:#6b7280;font-weight:700'>BALDEAÇÃO</span>"

            else:

                marcador = "<span style='font-size:14px'>●</span>"

                rotulo = ""

            estacoes_html.append(

                f"<div style='display:flex;align-items:center;gap:10px;min-height:30px;'>"

                f"<div style='width:18px;text-align:center;color:{cor};z-index:2;background:white'>{marcador}</div>"

                f"<div style='font-size:13px;color:#1f2937'><b>{html.escape(estacao)}</b>{rotulo}</div>"

                f"</div>"

            )

        blocos.append(

            f"<div style='background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:14px 16px;margin:10px 0;'>"

            f"<div style='display:flex;align-items:center;gap:9px;margin-bottom:10px;'>"

            f"<span style='width:12px;height:12px;border-radius:50%;background:{cor};display:inline-block'></span>"

            f"<b style='color:{cor};font-size:14px'>{html.escape(linha)}</b>"

            f"<span style='font-size:11px;color:#6b7280'>{len(estacoes)-1} trecho(s)</span>"

            f"</div>"

            f"<div style='position:relative;padding-left:2px'>"

            f"<div style='position:absolute;left:10px;top:14px;bottom:14px;width:4px;background:{cor};border-radius:4px;opacity:.9'></div>"

            + "".join(estacoes_html) +

            "</div></div>"

        )

    return (

        "<div style='margin-top:16px;font-family:Arial,sans-serif'>"

        "<div style='font-size:17px;font-weight:700;color:#1f2937;margin-bottom:8px'>Mini mapa da rota</div>"

        "<div style='font-size:12px;color:#6b7280;margin-bottom:10px'>Percurso separado por linha</div>"

        + "".join(blocos) +

        "</div>"

    )

def desenhar_linhas_html(resultado=None):

    rota = set(resultado["caminho"] or []) if resultado else set()

    visitados = set(resultado["visitados"] or []) if resultado else set()

    bloqueadas = set(resultado["bloqueadas"] or []) if resultado else set()

    blocos = []

    for linha, estacoes in LINHAS.items():

        cor = CORES[linha]

        itens = []

        for i, estacao in enumerate(estacoes):

            bg = _cor_estacao(estacao, rota, visitados, bloqueadas)

            fg = "#ffffff" if bg in {"#616161", "#ff9800", "#9e9e9e"} else "#111111"

            item = (

                f'<span style="display:inline-block;margin:3px;padding:5px 8px;'

                f'border-radius:12px;border:2px solid {cor};background:{bg};'

                f'color:{fg};font-family:Arial;font-size:12px;">'

                f'{html.escape(estacao)}</span>'

            )

            itens.append(item)

            if i < len(estacoes) - 1:

                itens.append(

                    f'<span style="color:{cor};font-weight:bold;"> ─ </span>'

                )

        blocos.append(

            f'<div style="margin:10px 0;"><b style="color:{cor};">{linha}</b><br>'

            + "".join(itens)

            + "</div>"

        )

    legenda = """

    <div style="font-family:Arial;font-size:12px;margin-top:12px;">

      <b>Legenda:</b>

      <span style="background:#ff9800;color:#fff;padding:3px 6px;border-radius:5px;">rota</span>

      <span style="background:#9e9e9e;color:#fff;padding:3px 6px;border-radius:5px;">visitada</span>

      <span style="background:#616161;color:#fff;padding:3px 6px;border-radius:5px;">bloqueada</span>

    </div>

    """

    return "<div>" + "".join(blocos) + legenda + "</div>"

def criar_interface():
    import ipywidgets as widgets
    from IPython.display import display, HTML
    import traceback

    estilo = widgets.HTML("""
    <style>
    .metro-header {
        background: linear-gradient(135deg, #1565c0, #1e88e5);
        color: white;
        padding: 25px 30px;
        border-radius: 14px 14px 0 0;
    }
    .metro-title {
        font-size: 28px;
        font-weight: bold;
        margin-bottom: 5px;
    }
    .metro-subtitle {
        font-size: 14px;
        opacity: 0.9;
    }
    .status-online {
        display: inline-block;
        background: rgba(255,255,255,0.18);
        padding: 7px 12px;
        border-radius: 20px;
        font-size: 12px;
        margin-top: 12px;
    }
    .section-title {
        font-size: 20px;
        font-weight: bold;
        color: #263238;
        margin: 10px 0 15px 0;
    }
    .card {
        background: white;
        border-radius: 12px;
        padding: 18px;
        box-shadow: 0 3px 12px rgba(0,0,0,0.07);
        border: 1px solid #e5e7eb;
    }
    .stat-number {
        font-size: 28px;
        font-weight: bold;
        color: #1565c0;
    }
    .stat-label {
        font-size: 12px;
        color: #6b7280;
        margin-top: 4px;
    }
    .line-card {
        background: white;
        border-radius: 10px;
        padding: 12px 15px;
        border-left: 6px solid;
        box-shadow: 0 2px 8px rgba(0,0,0,0.06);
    }
    .route-box {
        background: #ffffff;
        border-radius: 12px;
        padding: 18px;
        border: 1px solid #e1e5ea;
    }
    .station-origin {
        color: #2e7d32;
        font-weight: bold;
    }
    .station-destination {
        color: #d32f2f;
        font-weight: bold;
    }
    .alert-box {
        background: #fff8e1;
        border-left: 5px solid #ff9800;
        padding: 12px;
        border-radius: 7px;
    }
    .success-box {
        background: #e8f5e9;
        border-left: 5px solid #2e7d32;
        padding: 12px;
        border-radius: 7px;
    }
    .info-box {
        background: #e3f2fd;
        border-left: 5px solid #1e88e5;
        padding: 12px;
        border-radius: 7px;
    }
    </style>
    """)

    header = widgets.HTML("""
    <div class="metro-header">
        <div class="metro-title">🚇 MetrôBot SP 2.0</div>
        <div class="metro-subtitle">Planejador de rotas do Metrô de São Paulo</div>
        <div class="status-online">● Sistema operacional</div>
    </div>
    """)

    btn_inicio = widgets.Button(
        description="🏠  Início",
        layout=widgets.Layout(width="100%", height="42px")
    )

    btn_rotas = widgets.Button(
        description="🚇  Planejar rota",
        layout=widgets.Layout(width="100%", height="42px")
    )

    menu = widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-size:12px;font-weight:bold;color:#6b7280;"
                "margin:10px 5px;'>MENU</div>"
            ),
            btn_inicio,
            btn_rotas
        ],
        layout=widgets.Layout(
            width="190px",
            min_width="190px",
            padding="15px"
        )
    )

    total_estacoes = len(GRAFO)

    cards_inicio = widgets.HBox(
        [
            widgets.HTML(
                f"<div class='card'><div class='stat-number'>{total_estacoes}</div>"
                "<div class='stat-label'>Estações cadastradas</div></div>"
            ),
            widgets.HTML(
                f"<div class='card'><div class='stat-number'>{len(LINHAS)}</div>"
                "<div class='stat-label'>Linhas monitoradas</div></div>"
            ),
            widgets.HTML(
                "<div class='card'><div class='stat-number'>3</div>"
                "<div class='stat-label'>Estações de integração</div></div>"
            ),
            widgets.HTML(
                "<div class='card'><div class='stat-number'>2</div>"
                "<div class='stat-label'>Algoritmos disponíveis</div></div>"
            )
        ],
        layout=widgets.Layout(
            width="100%",
            gap="12px",
            flex_flow="row wrap"
        )
    )

    status_linhas = []
    for linha, cor in CORES.items():
        status_linhas.append(
            widgets.HTML(
                f"<div class='line-card' style='border-left-color:{cor};'>"
                f"<div style='font-weight:bold;color:{cor};font-size:15px;'>{linha}</div>"
                "<div style='color:#6b7280;font-size:12px;margin-top:5px;'>"
                "● Operação disponível</div></div>"
            )
        )

    pagina_inicio = widgets.VBox(
        [
            widgets.HTML("<div class='section-title'>Visão geral</div>"),
            cards_inicio,
            widgets.HTML("<div class='section-title' style='margin-top:24px;'>Status das linhas</div>"),
            widgets.VBox(status_linhas, layout=widgets.Layout(gap="8px")),
            widgets.HTML("""
            <div class="info-box" style="margin-top:20px;">
                <b>💡 Como utilizar</b><br><br>
                Você pode escrever um pedido como:<br>
                <b>"Estou na Sé e quero ir ao MASP"</b><br><br>
                O sistema identifica automaticamente a origem, o destino
                e necessidades de acessibilidade.
            </div>
            """)
        ],
        layout=widgets.Layout(width="100%")
    )

    entrada = widgets.Textarea(
        placeholder=(
            "Ex.: Estou na Sé e quero ir ao MASP\n"
            "Ex.: Quero ir da Sé para Jabaquara"
        ),
        layout=widgets.Layout(width="100%", height="90px")
    )

    opcoes_origem_destino = [("", None)]
    opcoes_origem_destino += [(nome, ("estacao", nome)) for nome in GRAFO]
    opcoes_origem_destino += [(nome, ("local", nome)) for nome in LOCAIS]

    origem = widgets.Dropdown(
        options=opcoes_origem_destino,
        description="Origem:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "80px"}
    )

    destino = widgets.Dropdown(
        options=opcoes_origem_destino,
        description="Destino:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "80px"}
    )

    acessibilidade = widgets.Checkbox(
        value=False,
        description="♿ Precisa de acessibilidade",
        indent=False
    )

    algoritmo = widgets.Dropdown(
        options=["BFS", "DFS"],
        value="BFS",
        description="Algoritmo:",
        layout=widgets.Layout(width="250px"),
        style={"description_width": "80px"}
    )

    fechadas = widgets.SelectMultiple(
        options=list(GRAFO.keys()),
        description="Estações:",
        layout=widgets.Layout(width="100%", height="140px"),
        style={"description_width": "80px"}
    )

    manutencao = widgets.SelectMultiple(
        options=list(GRAFO.keys()),
        description="Elevadores:",
        layout=widgets.Layout(width="100%", height="140px"),
        style={"description_width": "80px"}
    )

    paralisa = widgets.SelectMultiple(
        options=list(LINHAS.keys()),
        description="Linhas:",
        layout=widgets.Layout(width="100%", height="90px"),
        style={"description_width": "80px"}
    )

    btn_interpretar = widgets.Button(
        description="🧠 Interpretar pedido",
        button_style="info",
        layout=widgets.Layout(width="190px", height="42px")
    )

    btn_buscar = widgets.Button(
        description="🔎 Buscar rota",
        button_style="success",
        layout=widgets.Layout(width="160px", height="42px")
    )

    saida_mensagem = widgets.HTML()
    saida_rota = widgets.Output(
        layout=widgets.Layout(width="100%")
    )

    avancadas = widgets.Accordion(
        children=[
            widgets.VBox(
                [
                    widgets.HBox(
                        [fechadas, manutencao],
                        layout=widgets.Layout(width="100%", gap="15px")
                    ),
                    paralisa
                ],
                layout=widgets.Layout(width="100%", gap="10px")
            )
        ],
        layout=widgets.Layout(width="100%")
    )
    avancadas.set_title(0, "⚙️ Configurações avançadas")
    avancadas.selected_index = None

    pagina_rotas = widgets.VBox(
        [
            widgets.HTML("<div class='section-title'>🚇 Planejamento de rota</div>"),
            widgets.HTML("""
            <div class="info-box">
                Digite seu pedido em linguagem natural ou selecione manualmente
                a origem e o destino.
            </div>
            """),
            widgets.HTML("<b>Pedido em linguagem natural</b>"),
            entrada,
            widgets.HBox(
                [btn_interpretar, btn_buscar],
                layout=widgets.Layout(gap="10px", flex_flow="row wrap")
            ),
            saida_mensagem,
            widgets.HTML("<b>Origem e destino</b>"),
            origem,
            destino,
            widgets.HBox(
                [acessibilidade, algoritmo],
                layout=widgets.Layout(
                    align_items="center",
                    gap="30px",
                    flex_flow="row wrap"
                )
            ),
            avancadas,
            saida_rota
        ],
        layout=widgets.Layout(width="100%", gap="10px")
    )

    area_conteudo = widgets.VBox(
        [pagina_inicio],
        layout=widgets.Layout(
            width="100%",
            padding="25px",
            overflow="visible"
        )
    )

    def selecionar_menu(nome):
        if nome == "inicio":
            area_conteudo.children = (pagina_inicio,)
            btn_inicio.button_style = "info"
            btn_rotas.button_style = ""
        else:
            area_conteudo.children = (pagina_rotas,)
            btn_inicio.button_style = ""
            btn_rotas.button_style = "info"

    def mostrar_resultado(resultado):
        caminho = resultado["caminho"]

        display(HTML("""
        <div class="section-title" style="margin-top:10px;">
            ✓ Rota encontrada
        </div>
        """))

        cards = widgets.HBox(
            [
                widgets.HTML(
                    f"<div class='card'><div class='stat-number'>{resultado['paradas']}</div>"
                    "<div class='stat-label'>Paradas</div></div>"
                ),
                widgets.HTML(
                    f"<div class='card'><div class='stat-number'>{resultado['baldeacoes']}</div>"
                    "<div class='stat-label'>Baldeações</div></div>"
                ),
                widgets.HTML(
                    f"<div class='card'><div class='stat-number'>{resultado['tempo_min']} min</div>"
                    "<div class='stat-label'>Tempo estimado</div></div>"
                ),
                widgets.HTML(
                    f"<div class='card'><div class='stat-number'>{resultado['algoritmo']}</div>"
                    "<div class='stat-label'>Busca utilizada</div></div>"
                )
            ],
            layout=widgets.Layout(gap="10px", flex_flow="row wrap")
        )
        display(cards)

        display(HTML(
            f"<div class='route-box' style='margin-top:14px;'>"
            f"🟢 <span class='station-origin'>{html.escape(resultado['origem'])}</span>"
            "<div style='margin:10px 0;color:#9ca3af;'>↓</div>"
            f"🔴 <span class='station-destination'>{html.escape(resultado['destino'])}</span>"
            "</div>"
        ))

        caminho_html = []
        for i, estacao in enumerate(caminho):
            if i == 0:
                icone = "🟢"
            elif i == len(caminho) - 1:
                icone = "🔴"
            else:
                icone = "●"

            caminho_html.append(
                "<div style='padding:8px;border-left:3px solid #1e88e5;margin-left:8px;'>"
                f"{icone} <b>{html.escape(estacao)}</b></div>"
            )

        display(HTML(
            "<div class='card' style='margin-top:14px;'>"
            "<div style='font-size:17px;font-weight:bold;margin-bottom:12px;'>"
            "🧭 Itinerário</div>"
            + "".join(caminho_html)
            + "</div>"
        ))

        if resultado["onde_baldear"]:
            itens = []
            for estacao, linha in resultado["onde_baldear"]:
                itens.append(
                    "<div style='padding:8px;margin:5px 0;background:#f3f4f6;"
                    "border-radius:7px;'>"
                    f"🔄 <b>{html.escape(estacao)}</b> → {html.escape(linha)}</div>"
                )

            display(HTML(
                "<div class='card' style='margin-top:14px;'>"
                "<div style='font-size:17px;font-weight:bold;margin-bottom:10px;'>"
                "🔄 Baldeações</div>"
                + "".join(itens)
                + "</div>"
            ))

        if resultado["alertas"]:
            alertas_html = "<br>".join(
                f"⚠️ {html.escape(a)}" for a in resultado["alertas"]
            )
            display(HTML(
                "<div class='alert-box' style='margin-top:14px;'>"
                f"<b>Alertas</b><br><br>{alertas_html}</div>"
            ))

    def interpretar(_=None):
        texto = entrada.value.strip()
        saida_mensagem.value = ""

        if not texto:
            saida_mensagem.value = (
                "<div class='alert-box'>⚠️ Digite um pedido antes de interpretar.</div>"
            )
            return

        btn_interpretar.disabled = True
        btn_interpretar.description = "Interpretando..."

        try:
            pedido, mensagem = interpretar_pedido(texto)

            if pedido:
                origem.value = pedido["origem"]
                destino.value = pedido["destino"]
                acessibilidade.value = pedido["acessibilidade"]

                saida_mensagem.value = (
                    "<div class='success-box'><b>✓ Pedido interpretado</b><br><br>"
                    f"{html.escape(mensagem)}</div>"
                )
            else:
                saida_mensagem.value = (
                    f"<div class='alert-box'>⚠️ {html.escape(mensagem)}</div>"
                )

        except Exception as erro:
            saida_mensagem.value = (
                "<div class='alert-box'>⚠️ Erro ao interpretar pedido:<br>"
                f"{html.escape(str(erro))}</div>"
            )

        finally:
            btn_interpretar.disabled = False
            btn_interpretar.description = "🧠 Interpretar pedido"

    def buscar_rota(_=None):
        saida_mensagem.value = ""

        if not origem.value:
            saida_mensagem.value = (
                "<div class='alert-box'>⚠️ Selecione uma origem.</div>"
            )
            return

        if not destino.value:
            saida_mensagem.value = (
                "<div class='alert-box'>⚠️ Selecione um destino.</div>"
            )
            return

        btn_buscar.disabled = True
        btn_buscar.description = "Buscando..."

        try:
            resultado = planejar(
                pedido={
                    "origem": origem.value,
                    "destino": destino.value,
                    "acessibilidade": acessibilidade.value
                },
                fechadas=list(fechadas.value),
                manutencao=list(manutencao.value),
                algoritmo=algoritmo.value,
                linhas_paralisadas=list(paralisa.value)
            )

            saida_rota.clear_output(wait=True)

            with saida_rota:
                if resultado and resultado["caminho"]:
                    mostrar_resultado(resultado)

                    display(HTML(
                        "<div class='section-title' style='margin-top:18px;'>"
                        "🗺️ Mapa da rota"
                        "</div>"
                    ))

                    display(HTML(desenhar_mini_mapa_rota(resultado)))

                else:
                    display(HTML("""
                    <div class="alert-box">
                        <b>❌ Nenhuma rota encontrada.</b><br><br>
                        Verifique as estações bloqueadas, linhas paralisadas
                        ou tente outro algoritmo.
                    </div>
                    """))

        except Exception:
            saida_rota.clear_output(wait=True)
            erro = traceback.format_exc()

            with saida_rota:
                display(HTML(
                    "<div class='alert-box'><b>Erro ao calcular rota</b><br><br>"
                    + html.escape(erro).replace("\n", "<br>")
                    + "</div>"
                ))

        finally:
            btn_buscar.disabled = False
            btn_buscar.description = "🔎 Buscar rota"

    btn_inicio.on_click(lambda _: selecionar_menu("inicio"))
    btn_rotas.on_click(lambda _: selecionar_menu("rotas"))
    btn_interpretar.on_click(interpretar)
    btn_buscar.on_click(buscar_rota)

    principal = widgets.HBox(
        [menu, area_conteudo],
        layout=widgets.Layout(
            width="100%",
            align_items="stretch"
        )
    )

    sistema = widgets.VBox(
        [estilo, header, principal],
        layout=widgets.Layout(width="100%")
    )

    selecionar_menu("inicio")
    display(sistema)

criar_interface()

def _pedido_estacao(origem, destino, acessibilidade=False):

    return {

        "origem": ("estacao", origem),

        "destino": ("estacao", destino),

        "acessibilidade": acessibilidade,

    }

def rodar_testes():

    testes = []

    r = planejar(_pedido_estacao("Tucuruvi", "Corinthians-Itaquera"))

    assert r["paradas"] == 22

    assert r["baldeacoes"] == 1

    assert ("Sé", "Linha 3-Vermelha") in r["onde_baldear"]

    testes.append("Caso 1 OK")

    r = planejar(_pedido_estacao("Vila Madalena", "Jabaquara"))

    assert r["paradas"] == 14

    assert r["baldeacoes"] == 1

    assert r["onde_baldear"][0][0] in {"Paraíso", "Ana Rosa"}

    testes.append("Caso 2 OK")

    r = planejar(_pedido_estacao("Palmeiras-Barra Funda", "Vila Prudente"))

    assert r["paradas"] == 16

    assert r["baldeacoes"] == 2

    assert len(r["onde_baldear"]) == 2

    testes.append("Caso 3 OK")

    r = planejar(

        _pedido_estacao("Tucuruvi", "Brás"),

        fechadas=["Sé"],

    )

    assert r["caminho"] is None

    testes.append("Caso 4 OK")

    r = planejar(

        _pedido_estacao("Vila Madalena", "Jabaquara"),

        fechadas=["Paraíso"],

    )

    assert r["caminho"] is None

    testes.append("Caso 5 OK")

    r = planejar(

        _pedido_estacao("Vila Prudente", "Jabaquara"),

        fechadas=["Paraíso"],

    )

    assert r["paradas"] == 13

    assert r["caminho"][-1] == "Jabaquara"

    assert "Paraíso" not in r["caminho"]

    testes.append("Caso 6 OK")

    fatos, _ = encadear_para_frente(fatos_base())

    integracoes = {e for (e,) in consultar(fatos, "integracao")}

    assert integracoes == {"Sé", "Paraíso", "Ana Rosa"}

    testes.append("Extra 1 OK")

    r = planejar(

        _pedido_estacao("Vila Madalena", "Jabaquara"),

        linhas_paralisadas=["Linha 2-Verde"],

    )

    assert r["caminho"] is None

    testes.append("Extra 2 OK")

    print("TESTES PASSARAM:", len(testes))

    for t in testes:

        print("✓", t)

def comparar_bfs_dfs(viagens):

    print(f"{'Viagem':<42}{'Paradas':>8}{'BFS':>8}{'DFS':>8}")

    print("-" * 70)

    for origem, destino in viagens:

        c_bfs, v_bfs = bfs(GRAFO, origem, destino)

        c_dfs, v_dfs = dfs(GRAFO, origem, destino)

        print(

            f"{origem + ' → ' + destino:<42}"

            f"{len(c_bfs)-1 if c_bfs else '-':>8}"

            f"{len(v_bfs):>8}"

            f"{len(v_dfs):>8}"

        )


Declaração de uso da IA

a ia foi utilizada para melhorar a interface do sistema, corrigir bugs, problemas de estruturação do codigo (identação, sintaxe e etc) ou problemas de usabilidade da interface.